encontrar enlaces entre dos cadenas

In [ ]:
from Bio.PDB import PDBParser, NeighborSearch

def find_interactions(pdb_file, chain_id_1, chain_id_2, distance_cutoff=4.0):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein_complex', pdb_file)
    model = structure[0]

    # 1. Extraer los átomos de ambas cadenas
    atoms_1 = list(model[chain_id_1].get_atoms())
    atoms_2 = list(model[chain_id_2].get_atoms())

    # 2. Configurar la búsqueda por vecindad (NeighborSearch)
    # Usamos los átomos de la cadena 2 como referencia
    ns = NeighborSearch(atoms_2)
    
    interactions = set()

    # 3. Buscar átomos de la cadena 1 que estén cerca de la cadena 2
    for atom1 in atoms_1:
        # Encuentra átomos en atoms_2 a la distancia definida
        close_atoms = ns.search(atom1.coord, distance_cutoff)
        
        if close_atoms:
            res1 = atom1.get_parent()
            for atom2 in close_atoms:
                res2 = atom2.get_parent()
                
                # Guardamos la interacción de forma única (Residuo A - Residuo B)
                interaction = (
                    f"{res1.get_resname()}{res1.id[1]} (Ch:{chain_id_1})",
                    f"{res2.get_resname()}{res2.id[1]} (Ch:{chain_id_2})"
                )
                interactions.add(interaction)

    return interactions

# --- Configuración ---
archivo = "HDOCK_model_1_sin_NAG.pdb"  # Cambia por tu archivo
cadena_a = "A"
cadena_b = "B"
distancia = 3.0

contactos = find_interactions(archivo, cadena_a, cadena_b, distancia)

print(f"--- Contactos encontrados ({distancia} Å) ---")
for c1, c2 in sorted(contactos):
    print(f"{c1} <---> {c2}")

cuantas veces se repite cada enlace en el exel de las interacciones

In [ ]:
import pandas as pd
import numpy as np

# 1. Configuración
archivo_entrada = 'tu_archivo.xlsx'
fila_encabezado = 1  # (Recuerda: Fila Excel - 1)

print(f"📂 Cargando archivo: {archivo_entrada}...")
df = pd.read_excel(archivo_entrada, header=fila_encabezado)

# 2. Encontrar columnas
# Ordenamos (sorted) para asegurarnos que OppA columna 1 se une con THG columna 1, etc.
cols_oppa = sorted([c for c in df.columns if str(c).strip().startswith("OppA (A)")])
cols_thg = sorted([c for c in df.columns if str(c).strip().startswith("THG")])

print(f"✅ Se encontraron {len(cols_oppa)} pares de columnas.")

# Lista para guardar todas las interacciones
interacciones_lista = []

# 3. Procesar Parejas
for col_a, col_b in zip(cols_oppa, cols_thg):
    # Limpieza OppA (Texto, Mayúsculas, Sin espacios)
    val_a = df[col_a].astype(str).str.upper().str.strip()
    val_a = val_a.replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan})
    
    # Limpieza THG
    val_b = df[col_b].astype(str).str.upper().str.strip()
    val_b = val_b.replace({'NAN': np.nan, 'NONE': np.nan, '': np.nan})
    
    # Unir texto: "OPPA - THG"
    # Solo unimos si ambos existen (dropna)
    pares = (val_a + " - " + val_b).dropna()
    
    interacciones_lista.extend(pares.tolist())

# 4. Crear DataFrame con el Conteo
if interacciones_lista:
    # Convertimos la lista en un DataFrame
    df_res = pd.DataFrame(interacciones_lista, columns=['Interaccion'])
    
    # Contamos frecuencias
    conteo = df_res['Interaccion'].value_counts().reset_index()
    conteo.columns = ['Par_Interaccion', 'Frecuencia']
    
    print("\n" + "="*50)
    print(" REPORTE DE INTERACCIONES (Total)")
    print("="*50)
    print(conteo)
    
    # Guardar TODO (incluso las que salen 1 vez)
    conteo.to_excel("Frecuencia_Interacciones_COMPLETA.xlsx", index=False)
    print("\n✅ Archivo guardado: 'Frecuencia_Interacciones_COMPLETA.xlsx'")
    print(f"   Total de combinaciones únicas encontradas: {len(conteo)}")
else:
    print("❌ No se encontraron datos para analizar.")

detectar cuanto se repite cada aa en cada modelo OppA

In [15]:
import pandas as pd

# 1. Cargar el archivo
file_path = 'residuos.xlsx'
# Cargamos con todos los niveles de encabezado (ajusta según tu archivo, p.ej. [0,1,2,3,4])
df = pd.read_excel(file_path, header=[0, 1, 2, 3, 4,5])

# 2. Identificar pares de columnas (OppA y su respectiva THG)
# Buscamos los índices de las columnas que contienen 'OppA (A)'
oppa_indices = [i for i, col in enumerate(df.columns) if 'OppA (A)' in col]

# 3. Procesar datos para la matriz
resultados = {}

for idx in oppa_indices:
    # La columna THG (B) suele estar justo a la derecha (+1)
    col_oppa = df.columns[idx]
    col_thg = df.columns[idx + 1]
    
    # Nombre para la columna en el resultado final (p.ej. THG (B).1, .2...)
    nombre_col_resultado = f"OppA (A).{len(resultados)}" if len(resultados) > 0 else "OppA (A)"
    
    # Contar frecuencias de los datos en la columna OppA actual
    counts = df[col_oppa].value_counts()
    resultados[nombre_col_resultado] = counts

# 4. Crear el DataFrame final
# Al pasar el diccionario a DataFrame, pandas alinea automáticamente los índices (SER91, ALA17, etc.)
df_final = pd.DataFrame(resultados)

# 5. Limpieza final
df_final = df_final.fillna(0).astype(int) # Cambiar NaNs por 0 y convertir a enteros
df_final.index.name = None # Quitar nombre al índice para que se vea limpio

# 6. Guardar y mostrar
print(df_final)
df_final.to_excel('matriz_frecuencias.xlsx')

        OppA (A)  OppA (A).1  OppA (A).2  OppA (A).3  OppA (A).4  OppA (A).5  \
ALA282         0           0           0           0           0           0   
ALA287         0           0           0           0           0           0   
ASN2           2           2           0           2           2           0   
ASN421         1           1           0           1           1           0   
ASN53          0           0           0           0           0           0   
...          ...         ...         ...         ...         ...         ...   
Trp64          0           0           1           0           0           1   
VAL266         0           0           0           0           0           0   
VAL445         0           0           0           0           0           0   
VAL57          1           1           0           1           1           0   
VAL7           0           0           0           0           0           0   

        OppA (A).6  OppA (A).7  OppA (A

detectar cuanto se repite cada aa en cada modelo THG

In [17]:
import pandas as pd

# 1. Cargar el archivo
file_path = 'residuos.xlsx'
# Cargamos con todos los niveles de encabezado (ajusta según tu archivo, p.ej. [0,1,2,3,4])
df = pd.read_excel(file_path, header=[0, 1, 2, 3, 4, 5])

# 2. Identificar pares de columnas (OppA y su respectiva THG)
# Buscamos los índices de las columnas que contienen 'OppA (A)'
oppa_indices = [i for i, col in enumerate(df.columns) if 'THG (B)' in col]

# 3. Procesar datos para la matriz
resultados = {}

for idx in oppa_indices:
    # La columna THG (B) suele estar justo a la derecha (+1)
    col_oppa = df.columns[idx]
    col_thg = df.columns[idx + 1]
    
    # Nombre para la columna en el resultado final (p.ej. THG (B).1, .2...)
    nombre_col_resultado = f"THG (B).{len(resultados)}" if len(resultados) > 0 else "THG (B)"
    
    # Contar frecuencias de los datos en la columna OppA actual
    counts = df[col_oppa].value_counts()
    resultados[nombre_col_resultado] = counts

# 4. Crear el DataFrame final
# Al pasar el diccionario a DataFrame, pandas alinea automáticamente los índices (SER91, ALA17, etc.)
df_final = pd.DataFrame(resultados)

# 5. Limpieza final
df_final = df_final.fillna(0).astype(int) # Cambiar NaNs por 0 y convertir a enteros
df_final.index.name = None # Quitar nombre al índice para que se vea limpio

# 6. Guardar y mostrar
print(df_final)
df_final.to_excel('matriz_frecuencias THG.xlsx')

        THG (B)  THG (B).1  THG (B).2  THG (B).3  THG (B).4  THG (B).5  \
ALA144        0          0          0          0          0          0   
ALA17         2          2          0          2          2          0   
ALA22         1          1          0          1          1          0   
ALA507        0          0          0          0          0          0   
ALA7          0          0          0          0          0          0   
...         ...        ...        ...        ...        ...        ...   
Thr427        0          0          0          0          0          0   
Thr512        0          0          0          0          0          0   
Tyr328        0          0          0          0          0          0   
Tyr56         0          0          1          0          0          1   
VAL313        0          0          0          0          0          0   

        THG (B).6  THG (B).7  THG (B).8  THG (B).9  THG (B).10  THG (B).11  \
ALA144          1          1     

detectar puentes salinos

In [ ]:
from Bio.PDB import PDBParser
import numpy as np

pdb_file = "GRAMMX_receptor-ligand_model1_sin_NAG.pdb"
chain_A = "A"   # proteína 1
chain_B = "C"   # proteína 2
cutoff = 4.0

neg_res = {
    "ASP": ["OD1", "OD2"],
    "GLU": ["OE1", "OE2"]
}

pos_res = {
    "LYS": ["NZ"],
    "ARG": ["NH1", "NH2", "NE"],
    "HIS": ["ND1", "NE2"]
}

parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", pdb_file)
model = structure[0]

def get_atoms(chain, res_dict):
    atoms = []
    for res in chain:
        if res.get_resname() in res_dict:
            for atom_name in res_dict[res.get_resname()]:
                if atom_name in res:
                    atoms.append((res, res[atom_name]))
    return atoms

chain1 = model[chain_A]
chain2 = model[chain_B]

neg_atoms = get_atoms(chain1, neg_res) + get_atoms(chain2, neg_res)
pos_atoms = get_atoms(chain1, pos_res) + get_atoms(chain2, pos_res)

print("Puentes salinos detectados:\n")

for neg_resi, neg_atom in neg_atoms:
    for pos_resi, pos_atom in pos_atoms:
        if neg_resi.get_parent().id != pos_resi.get_parent().id:
            dist = np.linalg.norm(neg_atom.coord - pos_atom.coord)
            if dist <= cutoff:
                print(
                    f"{neg_resi.get_resname()} {neg_resi.id[1]} "
                    f"({neg_resi.get_parent().id})  <-->  "
                    f"{pos_resi.get_resname()} {pos_resi.id[1]} "
                    f"({pos_resi.get_parent().id})  :  {dist:.2f} Å"
                )

detectar puentes de hidrogeno

In [ ]:
from Bio.PDB import PDBParser
import numpy as np

pdb_file = "GRAMMX_receptor-ligand_model1_sin_NAG.pdb"
chain_A = "A"
chain_B = "C"
cutoff = 5.0

# Residuos hidrofóbicos
hydrophobic_res = [
    "ALA", "VAL", "LEU", "ILE",
    "MET", "PHE", "TRP", "PRO"
]

parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", pdb_file)
model = structure[0]

def get_hydrophobic_atoms(chain):
    atoms = []
    for res in chain:
        if res.get_resname() in hydrophobic_res:
            for atom in res:
                # excluir backbone
                if atom.get_name() not in ["N", "CA", "C", "O"]:
                    atoms.append((res, atom))
    return atoms

chain1 = model[chain_A]
chain2 = model[chain_B]

hydro_atoms_1 = get_hydrophobic_atoms(chain1)
hydro_atoms_2 = get_hydrophobic_atoms(chain2)

print("Interacciones hidrofóbicas detectadas:\n")

for res1, atom1 in hydro_atoms_1:
    for res2, atom2 in hydro_atoms_2:
        dist = np.linalg.norm(atom1.coord - atom2.coord)
        if dist <= cutoff:
            print(
                f"{res1.get_resname()} {res1.id[1]} ({chain_A})  <-->  "
                f"{res2.get_resname()} {res2.id[1]} ({chain_B})  :  {dist:.2f} Å"
            )

visualizar DM

In [ ]:
import py3Dmol

# 1. Leer el archivo PDB
with open("output.pdb", "r") as f:
    pdb_data = f.read()

# 2. Configurar la vista
# Importante: 'multimodel=True' permite que py3Dmol separe los frames
view = py3Dmol.view(width=800, height=600)
view.addModelsAsFrames(pdb_data, "pdb") 

# 3. Estilo
view.setStyle({'cartoon': {'color': 'spectrum'}})

# 4. Configurar la Animación
# animate({'loop': 'forward', 'step': 1}) hace que se mueva automáticamente
view.animate({'loop': 'forward', 'reps': 0}) 

view.zoomTo()
view.show()

analizar dartos de DM

In [ ]:
import mdtraj as md
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Definir los archivos de entrada
# Usa el archivo PDB final o inicial de tu dinámica como topología
archivo_topologia = 'output.pdb' 
archivo_trayectoria = 'trayectoria.dcd'

print("Cargando la trayectoria...")
traj = md.load(archivo_trayectoria, top=archivo_topologia)
print(f"Trayectoria cargada con éxito: {traj.n_frames} fotogramas y {traj.n_atoms} átomos.")

# ==========================================
# 2. Análisis de RMSD (Estabilidad Global)
# ==========================================
print("\nCalculando RMSD...")
# Alinear todos los fotogramas al primero (frame 0) para eliminar traslaciones/rotaciones
traj.superpose(traj, 0)
rmsd_valores = md.rmsd(traj, traj, 0)

# Graficar y guardar imagen
plt.figure(figsize=(8, 5))
plt.plot(rmsd_valores, color='blue', linewidth=1.5)
plt.xlabel('Fotograma')
plt.ylabel('RMSD (nm)')
plt.title('Estabilidad Global (RMSD)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('grafico_rmsd.png', dpi=300, bbox_inches='tight')
plt.close()

# Guardar tabla de datos
pd.DataFrame({'Fotograma': range(len(rmsd_valores)), 'RMSD_nm': rmsd_valores}).to_csv('datos_rmsd.csv', index=False)


# ==========================================
# 3. Análisis de RMSF (Flexibilidad Local)
# ==========================================
print("Calculando RMSF...")
# Extraer solo los Carbonos Alfa (CA) para medir flexibilidad del esqueleto
ca_indices = traj.topology.select('name CA')
traj_ca = traj.atom_slice(ca_indices)
traj_ca.superpose(traj_ca, 0)
rmsf_valores = md.rmsf(traj_ca, traj_ca, 0)

# Obtener los números reales de los residuos para el eje X
residuos = [atom.residue.resSeq for atom in traj.topology.atoms if atom.index in ca_indices]

# Graficar y guardar imagen
plt.figure(figsize=(10, 5))
plt.plot(residuos, rmsf_valores, color='red', linewidth=1.5)
plt.xlabel('Número de Residuo')
plt.ylabel('RMSF (nm)')
plt.title('Flexibilidad Local (RMSF)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('grafico_rmsf.png', dpi=300, bbox_inches='tight')
plt.close()

# Guardar tabla de datos
pd.DataFrame({'Residuo': residuos, 'RMSF_nm': rmsf_valores}).to_csv('datos_rmsf.csv', index=False)


# ==========================================
# 4. Análisis del Radio de Giro (Compacidad)
# ==========================================
print("Calculando Radio de Giro...")
# Calcular solo para la proteína (ignora agua/iones si los hay)
proteina_indices = traj.topology.select('protein')
traj_proteina = traj.atom_slice(proteina_indices)
rg_valores = md.compute_rg(traj_proteina)

# Graficar y guardar imagen
plt.figure(figsize=(8, 5))
plt.plot(rg_valores, color='green', linewidth=1.5)
plt.xlabel('Fotograma')
plt.ylabel('Radio de Giro (nm)')
plt.title('Compacidad (Radio de Giro)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('grafico_rg.png', dpi=300, bbox_inches='tight')
plt.close()

# Guardar tabla de datos
pd.DataFrame({'Fotograma': range(len(rg_valores)), 'Rg_nm': rg_valores}).to_csv('datos_rg.csv', index=False)

print("\n¡Análisis completado con éxito!")
print("Revisa tu carpeta, se han generado las imágenes (.png) y los archivos de datos (.csv).")